# Solutions — TPC-DS Retail Analytics (Hard 04)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

store_sales = spark.table("samples.tpcds_sf1.store_sales")
date_dim    = spark.table("samples.tpcds_sf1.date_dim")
item        = spark.table("samples.tpcds_sf1.item")
store       = spark.table("samples.tpcds_sf1.store")
customer    = spark.table("samples.tpcds_sf1.customer")


## Solution 1 — Year-over-Year Revenue Growth by Item Category

In [ ]:
w_lag = Window.partitionBy("i_category").orderBy("d_year")

result_1 = (
    store_sales
    .join(date_dim, F.col("ss_sold_date_sk") == F.col("d_date_sk"))
    .join(item,     F.col("ss_item_sk")      == F.col("i_item_sk"))
    .groupBy("i_category", "d_year")
    .agg(F.round(F.sum("ss_net_paid"), 2).alias("total_revenue"))
    .withColumn("prev_year_revenue", F.lag("total_revenue").over(w_lag))
    .withColumn("yoy_growth_pct",
        F.when(
            F.col("prev_year_revenue").isNotNull() & (F.col("prev_year_revenue") != 0),
            F.round((F.col("total_revenue") - F.col("prev_year_revenue")) / F.col("prev_year_revenue") * 100, 2)
        )
    )
    .orderBy("i_category", "d_year")
)


## Solution 2 — Holiday vs Non-Holiday Sales Comparison

In [ ]:
result_2 = (
    store_sales
    .join(date_dim, F.col("ss_sold_date_sk") == F.col("d_date_sk"))
    .withColumn("is_holiday", F.col("d_holiday") == "Y")
    .groupBy("is_holiday")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("total_revenue"),
        F.round(F.avg("ss_net_paid"), 2).alias("avg_net_paid"),
        F.count("*").alias("transaction_count"),
    )
    .orderBy("is_holiday")
)


## Solution 3 — Store Performance Ranking

In [ ]:
w = Window.orderBy(F.col("total_net_paid").desc())

result_3 = (
    store_sales
    .join(store, F.col("ss_store_sk") == F.col("s_store_sk"))
    .groupBy("s_store_name", "s_city", "s_state")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("total_net_paid"),
        F.round(
            (F.sum("ss_net_paid") - F.sum("ss_net_paid_inc_tax")) / F.sum("ss_net_paid") * 100,
            2
        ).alias("avg_margin_pct"),
    )
    .withColumn("revenue_rank", F.rank().over(w))
    .orderBy("revenue_rank")
)


## Solution 4 — Promotion Effectiveness

In [ ]:
result_4 = (
    store_sales
    .join(item, F.col("ss_item_sk") == F.col("i_item_sk"))
    .withColumn("is_promo", F.col("ss_promo_sk").isNotNull())
    .groupBy("i_category", "is_promo")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("revenue"),
        F.round(F.avg("ss_sales_price"), 2).alias("avg_price"),
    )
    .groupBy("i_category")
    .pivot("is_promo", [True, False])
    .agg(
        F.first("revenue").alias("revenue"),
        F.first("avg_price").alias("avg_price"),
    )
    .toDF("i_category",
          "promo_revenue", "promo_avg_price",
          "non_promo_revenue", "non_promo_avg_price")
    .orderBy("i_category")
)


## Solution 5 — Top Brands per Quarter by Revenue

In [ ]:
w = Window.partitionBy("d_quarter_name").orderBy(F.col("total_revenue").desc())

result_5 = (
    store_sales
    .join(date_dim, F.col("ss_sold_date_sk") == F.col("d_date_sk"))
    .join(item,     F.col("ss_item_sk")      == F.col("i_item_sk"))
    .groupBy("d_quarter_name", "i_brand")
    .agg(F.round(F.sum("ss_net_paid"), 2).alias("total_revenue"))
    .withColumn("brand_rank", F.rank().over(w))
    .filter(F.col("brand_rank") <= 5)
    .orderBy("d_quarter_name", "brand_rank")
)


## Solution 6 — Weekend vs Weekday Spending by Category

In [ ]:
result_6 = (
    store_sales
    .join(date_dim, F.col("ss_sold_date_sk") == F.col("d_date_sk"))
    .join(item,     F.col("ss_item_sk")      == F.col("i_item_sk"))
    .withColumn("is_weekend", F.col("d_day_name").isin("Saturday", "Sunday"))
    .groupBy("i_category", "is_weekend")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("total_revenue"),
        F.round(F.avg("ss_net_paid"), 2).alias("avg_net_paid"),
        F.count("*").alias("transaction_count"),
    )
    .orderBy("i_category", "is_weekend")
)


## Solution 7 — Customer Age Segment Analysis

In [ ]:
result_7 = (
    store_sales
    .join(customer, F.col("ss_customer_sk") == F.col("c_customer_sk"))
    .filter(F.col("c_birth_year").isNotNull())
    .withColumn("age", F.lit(2003) - F.col("c_birth_year"))  # TPC-DS reference year ~2003
    .withColumn("age_bucket",
        F.when(F.col("age") < 30, "Under 30")
         .when(F.col("age") < 45, "30-44")
         .when(F.col("age") < 60, "45-59")
         .otherwise("60+")
    )
    .groupBy("age_bucket")
    .agg(
        F.countDistinct("c_customer_sk").alias("customer_count"),
        F.round(F.sum("ss_net_paid"), 2).alias("total_spend"),
        F.round(F.avg("ss_net_paid"), 2).alias("avg_spend"),
    )
    .orderBy("age_bucket")
)


## Solution 8 — Loss-Making Store-Item Combinations

In [ ]:
result_8 = (
    store_sales
    .join(store, F.col("ss_store_sk") == F.col("s_store_sk"))
    .join(item,  F.col("ss_item_sk")  == F.col("i_item_sk"))
    .groupBy("s_store_name", "i_product_name", "i_category")
    .agg(
        F.round(F.sum("ss_net_profit"), 2).alias("total_loss"),
        F.count("*").alias("transaction_count"),
    )
    .filter(F.col("total_loss") < 0)
    .orderBy("total_loss")
)
